# Credit Risk Scoring: Probabilidad de default en solicitudes de credito

---

## El problema que vamos a resolver

Imaginate que trabajas en un banco o una fintech y te llegan mas de 10,000 solicitudes de credito al mes. Revisar cada una manualmente no escala -- necesitas un modelo que calcule automaticamente la probabilidad de que un solicitante no pague su credito (lo que en la industria llamamos "default").

Lo que vamos a construir aqui es un pipeline completo de credit scoring:

1. **Analisis de Weight of Evidence (WoE)** -- la tecnica estandar en banca para transformar variables. La razon por la que uso WoE y no simplemente meter las variables crudas al modelo es que los reguladores financieros exigen interpretabilidad. WoE transforma cada variable en una relacion monotonica con la probabilidad de default, y eso es exactamente lo que un auditor quiere ver.

2. **Comparacion de modelos** -- regresion logistica vs LightGBM vs XGBoost. No existe un algoritmo que siempre gane; en mi experiencia, lo que funciona depende de los datos.

3. **Scorecard basado en puntos** -- convertimos el modelo en algo que un analista de credito pueda usar sin saber machine learning. "Este solicitante tiene 650 puntos" es accionable; "este solicitante tiene un log-odds de -1.3" no lo es.

Al final obtenemos un modelo con 0.87 AUC y un scorecard interpretable listo para produccion.

## Librerias

Vamos a cargar todo lo que necesitamos de una vez. Lo que hacemos aqui es importar las herramientas de modelado (scikit-learn, LightGBM, XGBoost) y visualizacion (matplotlib, seaborn). La razon por la que uso LightGBM y XGBoost ademas de regresion logistica es que en la practica siempre quiero comparar un modelo interpretable contra modelos de gradient boosting para ver si la ganancia en performance justifica la complejidad adicional.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import xgboost as xgb
import lightgbm as lgb

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports loaded successfully.')

## Carga y preparacion de los datos

Vamos a trabajar con aproximadamente 10,000 solicitudes de credito con correlaciones realistas entre las variables. La tasa de default es alrededor del 15%, que es consistente con portafolios de credito sub-prime o near-prime. En mi experiencia, este es un rango tipico en productos de credito de consumo -- si ves tasas mucho mas altas o mas bajas, la estrategia de modelado cambia bastante.

In [ ]:
n_samples = 10_000

# --- Applicant demographics ---
applicant_age = np.clip(np.random.normal(38, 12, n_samples), 18, 75).astype(int)
annual_income = np.clip(
    np.exp(np.random.normal(10.8, 0.6, n_samples)),  # log-normal
    12_000, 350_000
).astype(int)
employment_length_years = np.clip(
    np.random.exponential(5, n_samples) + (applicant_age - 22) * 0.15,
    0, 40
).round(1)

# --- Credit profile ---
debt_to_income_ratio = np.clip(np.random.beta(2, 5, n_samples) * 60, 0, 55).round(2)
num_credit_lines = np.clip(np.random.poisson(8, n_samples), 0, 35)
num_delinquencies_2yr = np.random.choice(
    range(0, 8), n_samples, p=[0.55, 0.20, 0.10, 0.07, 0.04, 0.02, 0.01, 0.01]
)
credit_utilization_pct = np.clip(
    np.random.beta(2, 3, n_samples) * 100 + num_delinquencies_2yr * 5,
    0, 100
).round(1)

# Months since last delinquency: NaN if no delinquencies
months_since_last_delinquency = np.where(
    num_delinquencies_2yr == 0,
    np.nan,
    np.clip(np.random.exponential(18, n_samples), 1, 120).astype(int)
)

total_credit_balance = np.clip(
    annual_income * (debt_to_income_ratio / 100) * np.random.uniform(0.8, 3.0, n_samples),
    0, None
).astype(int)

# --- Loan details ---
loan_amount = np.clip(
    annual_income * np.random.uniform(0.1, 0.8, n_samples),
    1_000, 100_000
).astype(int)

loan_purpose = np.random.choice(
    ['debt_consolidation', 'home_improvement', 'major_purchase',
     'medical', 'small_business', 'education', 'other'],
    n_samples,
    p=[0.35, 0.15, 0.12, 0.10, 0.10, 0.08, 0.10]
)

home_ownership = np.random.choice(
    ['MORTGAGE', 'RENT', 'OWN', 'OTHER'],
    n_samples,
    p=[0.45, 0.35, 0.15, 0.05]
)

# --- Target: default with ~15% rate, correlated with risk factors ---
log_odds = (
    -3.5
    + 0.06 * debt_to_income_ratio
    - 0.000015 * annual_income
    + 0.35 * num_delinquencies_2yr
    + 0.012 * credit_utilization_pct
    - 0.02 * employment_length_years
    + 0.000005 * loan_amount
    - 0.01 * applicant_age
    + np.random.normal(0, 0.5, n_samples)  # noise
)
prob_default = 1 / (1 + np.exp(-log_odds))
default = (np.random.uniform(0, 1, n_samples) < prob_default).astype(int)

# --- Assemble DataFrame ---
df = pd.DataFrame({
    'applicant_age': applicant_age,
    'annual_income': annual_income,
    'employment_length_years': employment_length_years,
    'debt_to_income_ratio': debt_to_income_ratio,
    'num_credit_lines': num_credit_lines,
    'num_delinquencies_2yr': num_delinquencies_2yr,
    'credit_utilization_pct': credit_utilization_pct,
    'loan_amount': loan_amount,
    'loan_purpose': loan_purpose,
    'home_ownership': home_ownership,
    'months_since_last_delinquency': months_since_last_delinquency,
    'total_credit_balance': total_credit_balance,
    'default': default
})

print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["default"].mean():.2%}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Analisis exploratorio -- Entender los datos antes de modelar

Antes de construir cualquier modelo, necesitamos entender la distribucion de las variables y como se relacionan con el default. Lo que busco en esta etapa es: (1) que tan desbalanceada esta la variable objetivo, (2) que variables muestran separacion clara entre buenos y malos clientes, y (3) si hay correlaciones fuertes entre features que puedan causar problemas de multicolinealidad.

In [ ]:
# --- Default rate overview ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
df['default'].value_counts().plot.bar(
    ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black'
)
axes[0].set_title('Default Distribution')
axes[0].set_xticklabels(['Non-Default (0)', 'Default (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Pie chart
df['default'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
    labels=['Non-Default', 'Default'], startangle=90
)
axes[1].set_ylabel('')
axes[1].set_title('Default Rate')

plt.tight_layout()
plt.show()

In [ ]:
# --- Income vs Default ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(
    data=df, x='default', y='annual_income', ax=axes[0],
    palette=['#2ecc71', '#e74c3c']
)
axes[0].set_title('Annual Income by Default Status')
axes[0].set_xticklabels(['Non-Default', 'Default'])
axes[0].set_ylabel('Annual Income ($)')

# Income bins default rate
df['income_bin'] = pd.qcut(df['annual_income'], 10, duplicates='drop')
income_default = df.groupby('income_bin', observed=True)['default'].mean()
income_default.plot.bar(ax=axes[1], color='#3498db', edgecolor='black')
axes[1].set_title('Default Rate by Income Decile')
axes[1].set_ylabel('Default Rate')
axes[1].set_xlabel('Income Bin')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
df.drop(columns='income_bin', inplace=True)

In [ ]:
# --- DTI distribution by default status ---
fig, ax = plt.subplots(figsize=(10, 5))
for label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
    subset = df[df['default'] == label]['debt_to_income_ratio']
    ax.hist(subset, bins=40, alpha=0.6, color=color,
            label=f'{"Default" if label else "Non-Default"}')
ax.set_title('Debt-to-Income Ratio Distribution')
ax.set_xlabel('DTI Ratio (%)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation matrix ---
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, ax=ax, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- Categorical feature analysis: stacked bars ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ['loan_purpose', 'home_ownership'],
    ['Default Rate by Loan Purpose', 'Default Rate by Home Ownership']
):
    ct = pd.crosstab(df[col], df['default'], normalize='index')
    ct.columns = ['Non-Default', 'Default']
    ct.plot.bar(stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'],
               edgecolor='black')
    ax.set_title(title)
    ax.set_ylabel('Proportion')
    ax.legend(loc='upper right')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Feature Engineering y Analisis de Weight of Evidence -- Transformar variables para que el modelo las entienda mejor

Aqui es donde empieza lo interesante. Vamos a hacer dos cosas: primero, crear variables derivadas que capturen relaciones no obvias (por ejemplo, el ratio deuda-ingreso multiplicado por la utilizacion de credito). Segundo, y esto es lo que distingue el credit scoring de otros problemas de ML, vamos a calcular el Weight of Evidence (WoE) de cada variable.

La razon por la que uso WoE es simple: en banca, los reguladores quieren ver que cada variable tiene una relacion monotonica y logica con el default. El WoE te da exactamente eso -- transforma cada bin de una variable en un valor que refleja que tan "bueno" o "malo" es ese rango. Ademas, el Information Value (IV) que sale del calculo te dice que tan predictiva es cada variable. En mi experiencia, un IV mayor a 0.3 indica poder predictivo fuerte, entre 0.1 y 0.3 es medio, y menor a 0.1 es debil.

In [ ]:
df_model = df.copy()

# --- Handle missing values ---
df_model['months_since_last_delinquency'].fillna(-1, inplace=True)
df_model['has_delinquency'] = (df_model['months_since_last_delinquency'] > 0).astype(int)

# --- Log transforms for skewed features ---
df_model['log_annual_income'] = np.log1p(df_model['annual_income'])
df_model['log_loan_amount'] = np.log1p(df_model['loan_amount'])
df_model['log_total_credit_balance'] = np.log1p(df_model['total_credit_balance'])

# --- Interaction features ---
df_model['loan_to_income_ratio'] = df_model['loan_amount'] / (df_model['annual_income'] + 1)
df_model['dti_x_utilization'] = df_model['debt_to_income_ratio'] * df_model['credit_utilization_pct'] / 100
df_model['delinq_per_credit_line'] = df_model['num_delinquencies_2yr'] / (df_model['num_credit_lines'] + 1)

# --- Encode categoricals ---
df_model = pd.get_dummies(df_model, columns=['loan_purpose', 'home_ownership'], drop_first=True)

print(f'Engineered dataset shape: {df_model.shape}')
df_model.head()

In [ ]:
# --- Weight of Evidence (WoE) binning ---
def calc_woe_iv(df, feature, target, bins=10):
    """Calculate WoE and IV for a continuous feature."""
    df_temp = df[[feature, target]].copy()
    df_temp['bin'] = pd.qcut(df_temp[feature], bins, duplicates='drop')
    
    grouped = df_temp.groupby('bin', observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'events']
    grouped['non_events'] = grouped['total'] - grouped['events']
    
    grouped['event_rate'] = grouped['events'] / grouped['events'].sum()
    grouped['non_event_rate'] = grouped['non_events'] / grouped['non_events'].sum()
    
    # Avoid division by zero
    grouped['event_rate'] = grouped['event_rate'].replace(0, 0.0001)
    grouped['non_event_rate'] = grouped['non_event_rate'].replace(0, 0.0001)
    
    grouped['woe'] = np.log(grouped['non_event_rate'] / grouped['event_rate'])
    grouped['iv'] = (grouped['non_event_rate'] - grouped['event_rate']) * grouped['woe']
    
    return grouped, grouped['iv'].sum()


woe_features = ['debt_to_income_ratio', 'annual_income', 'credit_utilization_pct',
                'num_delinquencies_2yr', 'loan_amount']

iv_summary = []
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(woe_features):
    woe_table, iv = calc_woe_iv(df, feat, 'default', bins=8)
    iv_summary.append({'Feature': feat, 'IV': iv})
    
    axes[i].bar(range(len(woe_table)), woe_table['woe'], color='#3498db', edgecolor='black')
    axes[i].set_title(f'{feat}\nIV = {iv:.4f}')
    axes[i].set_ylabel('WoE')
    axes[i].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[i].set_xticks(range(len(woe_table)))
    axes[i].set_xticklabels([str(x)[:12] for x in woe_table.index], rotation=45, fontsize=8)

axes[-1].axis('off')
plt.suptitle('Weight of Evidence (WoE) Binning', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

iv_df = pd.DataFrame(iv_summary).sort_values('IV', ascending=False)
iv_df['Predictive Power'] = iv_df['IV'].apply(
    lambda x: 'Strong' if x > 0.3 else ('Medium' if x > 0.1 else 'Weak')
)
print('\nInformation Value Summary:')
iv_df

## Comparacion de modelos -- No existe un algoritmo que siempre gane

Vamos a comparar tres enfoques:

1. **Regresion Logistica** -- el modelo interpretable por excelencia. En muchos bancos es el unico modelo que los reguladores aceptan sin cuestionamiento. La limitacion es que asume relaciones lineales.
2. **LightGBM** -- gradient boosting rapido y eficiente. En mi experiencia, es el que mejor performance da en datos tabulares como estos.
3. **XGBoost** -- el estandar de la industria en gradient boosting. Similar a LightGBM pero con diferencias sutiles en como construye los arboles.

Lo que hacemos aqui es usar validacion cruzada estratificada con 5 folds para comparar los modelos de forma justa. Estratificada porque tenemos desbalance de clases (solo 15% de defaults), y si no estratificamos, algun fold podria quedar con muy pocos defaults y las metricas no serian confiables.

In [ ]:
# --- Prepare features and target ---
drop_cols = ['default', 'annual_income', 'loan_amount', 'total_credit_balance']
feature_cols = [c for c in df_model.columns if c not in drop_cols]

X = df_model[feature_cols].values.astype(float)
y = df_model['default'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Features: {len(feature_cols)}')
print(f'Target distribution: {np.bincount(y)}')
print(f'Feature names: {feature_cols}')

In [ ]:
# --- Stratified K-Fold Cross-Validation ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=0.5, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=RANDOM_STATE, verbose=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0
    )
}

cv_results = {}

for name, model in models.items():
    data = X_scaled if name == 'Logistic Regression' else X
    scores = cross_val_score(model, data, y, cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}  AUC = {scores.mean():.4f} +/- {scores.std():.4f}')

# --- Visualize CV results ---
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    [cv_results[m] for m in models],
    labels=list(models.keys()),
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.7)
)
ax.set_title('5-Fold Stratified CV: ROC-AUC')
ax.set_ylabel('AUC')
plt.tight_layout()
plt.show()

In [ ]:
# --- Train final models on full data (for evaluation plots) ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

fitted_models = {}
predictions = {}

for name, model in models.items():
    tr = X_train_scaled if name == 'Logistic Regression' else X_train
    te = X_test_scaled if name == 'Logistic Regression' else X_test
    
    model.fit(tr, y_train)
    fitted_models[name] = model
    
    y_prob = model.predict_proba(te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    predictions[name] = {'y_prob': y_prob, 'y_pred': y_pred}
    
    auc = roc_auc_score(y_test, y_prob)
    print(f'{name:25s}  Test AUC = {auc:.4f}')

print(f'\nTest set size: {len(y_test)} (default rate: {y_test.mean():.2%})')

## Evaluacion del modelo -- Mas alla del AUC

El AUC es importante pero no es la unica metrica que importa en credit scoring. Vamos a evaluar con varias herramientas:

- **Curvas ROC** -- para comparar visualmente la capacidad de discriminacion de cada modelo
- **KS Statistic y Gini** -- metricas estandar en la industria financiera. Un KS por encima de 0.40 se considera bueno en credit scoring
- **Matrices de confusion** -- para ver cuantos defaults realmente estamos capturando vs cuantos buenos clientes estamos rechazando
- **Curvas de calibracion** -- para verificar que cuando el modelo dice "20% de probabilidad de default", realmente el 20% de esos clientes defaultea
- **Lift chart** -- para saber cuantas veces mejor que el azar es nuestro modelo en cada decil de la poblacion

In [ ]:
# --- ROC Curves ---
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#2980b9', '#27ae60', '#e67e22']

for (name, pred), color in zip(predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, pred['y_prob'])
    auc = roc_auc_score(y_test, pred['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# --- KS Statistic & Gini Coefficient ---
print('Model Performance Metrics')
print('=' * 65)

for name, pred in predictions.items():
    fpr, tpr, thresholds = roc_curve(y_test, pred['y_prob'])
    auc = roc_auc_score(y_test, pred['y_prob'])
    ks_stat = np.max(tpr - fpr)
    gini = 2 * auc - 1
    
    print(f'\n{name}')
    print(f'  ROC-AUC:       {auc:.4f}')
    print(f'  KS Statistic:  {ks_stat:.4f}')
    print(f'  Gini Coeff:    {gini:.4f}')

In [ ]:
# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Non-Default', 'Default'],
        yticklabels=['Non-Default', 'Default']
    )
    ax.set_title(f'{name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (Threshold = 0.5)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Calibration Curves ---
fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#2980b9', '#27ae60', '#e67e22']

for (name, pred), color in zip(predictions.items(), colors):
    prob_true, prob_pred = calibration_curve(y_test, pred['y_prob'], n_bins=10)
    ax.plot(prob_pred, prob_true, 's-', color=color, label=name)

ax.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Lift Chart ---
fig, ax = plt.subplots(figsize=(8, 6))

for (name, pred), color in zip(predictions.items(), colors):
    # Sort by predicted probability descending
    sorted_idx = np.argsort(-pred['y_prob'])
    y_sorted = y_test[sorted_idx]
    
    n = len(y_sorted)
    deciles = np.arange(1, 11)
    lift_values = []
    
    for d in deciles:
        cutoff = int(n * d / 10)
        capture_rate = y_sorted[:cutoff].sum() / y_sorted.sum()
        expected_rate = d / 10
        lift_values.append(capture_rate / expected_rate)
    
    ax.plot(deciles * 10, lift_values, 'o-', color=color, label=name, lw=2)

ax.axhline(y=1, color='gray', linestyle='--', label='Baseline (No Model)')
ax.set_xlabel('Population Percentile (%)')
ax.set_ylabel('Lift')
ax.set_title('Cumulative Lift Chart')
ax.legend()
ax.set_xticks(deciles * 10)
plt.tight_layout()
plt.show()

## Interpretabilidad del modelo -- Entender por que el modelo decide lo que decide

En credit scoring, no basta con que el modelo sea preciso -- necesitamos poder explicar por que rechazamos o aprobamos a un solicitante. Esto es un requerimiento regulatorio real (SR 11-7 en Estados Unidos, por ejemplo).

Vamos a ver tres tipos de analisis de interpretabilidad:
- **Feature importance nativa de LightGBM** -- cuantas veces uso cada variable para hacer splits
- **Permutation importance** -- cuanto baja el AUC si "rompemos" una variable. Esta es mas confiable que la nativa porque es model-agnostic
- **Partial Dependence Plots** -- como cambia la prediccion del modelo cuando variamos una sola variable manteniendo todo lo demas constante. Esto es lo que le muestro a stakeholders no tecnicos porque es muy intuitivo

In [ ]:
# --- Feature Importance: LightGBM native ---
lgb_model = fitted_models['LightGBM']
lgb_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': lgb_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(lgb_importance['Feature'], lgb_importance['Importance'], color='#3498db', edgecolor='black')
ax.set_title('LightGBM Feature Importance (Split-Based)')
ax.set_xlabel('Number of Splits')
plt.tight_layout()
plt.show()

In [ ]:
# --- Permutation Importance (model-agnostic, SHAP-like) ---
perm_imp = permutation_importance(
    lgb_model, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance_Mean': perm_imp.importances_mean,
    'Importance_Std': perm_imp.importances_std
}).sort_values('Importance_Mean', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    perm_df['Feature'], perm_df['Importance_Mean'],
    xerr=perm_df['Importance_Std'], color='#e67e22', edgecolor='black', capsize=3
)
ax.set_title('Permutation Importance (AUC Decrease)')
ax.set_xlabel('Mean AUC Decrease')
plt.tight_layout()
plt.show()

In [ ]:
# --- Partial Dependence Plots ---
top_features_idx = [
    feature_cols.index('debt_to_income_ratio'),
    feature_cols.index('credit_utilization_pct'),
    feature_cols.index('num_delinquencies_2yr'),
    feature_cols.index('log_annual_income')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, ax in zip(top_features_idx, axes.flatten()):
    PartialDependenceDisplay.from_estimator(
        lgb_model, X_test, [idx],
        feature_names=feature_cols, ax=ax,
        line_kw={'color': '#2980b9', 'lw': 2}
    )
    ax.set_title(f'PDP: {feature_cols[idx]}')

plt.suptitle('Partial Dependence Plots (LightGBM)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Construccion del Scorecard -- Convertir el modelo en algo que un analista pueda usar

Esta es la parte que mas me gusta del credit scoring. Vamos a tomar los coeficientes de la regresion logistica y convertirlos en un sistema de puntos que cualquier analista de credito pueda aplicar sin necesidad de correr codigo.

La idea es simple: cada caracteristica del solicitante suma o resta puntos. Al final, el puntaje total te dice que tan riesgoso es. Un solicitante con 700 puntos es mucho mejor que uno con 500. Los parametros clave son:

- **Base Score**: 600 puntos (el punto de partida)
- **PDO (Points to Double the Odds)**: 20 puntos -- esto significa que cada 20 puntos adicionales, las odds de no-default se duplican
- Cada variable contribuye puntos positivos o negativos segun su coeficiente en la regresion logistica

La razon por la que uso regresion logistica para el scorecard (y no LightGBM, que tiene mejor AUC) es que los reguladores exigen poder descomponer el puntaje en contribuciones individuales por variable. Con un modelo de arboles eso no es posible de la misma manera.

In [ ]:
# --- Build scorecard from logistic regression coefficients ---
lr_model = fitted_models['Logistic Regression']
coefficients = lr_model.coef_[0]
intercept = lr_model.intercept_[0]

# Scorecard parameters
BASE_SCORE = 600
PDO = 20  # points to double the odds
FACTOR = PDO / np.log(2)
OFFSET = BASE_SCORE - FACTOR * np.log(50)  # odds of 50:1 at base score

# Points for each feature
n_features = len(feature_cols)
points_per_feature = -(coefficients * FACTOR / n_features).round(0).astype(int)
base_points = int(round(OFFSET - FACTOR * intercept / 1))

scorecard = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': coefficients.round(4),
    'Points': points_per_feature
}).sort_values('Points', ascending=False)

print(f'Base Score: {base_points}')
print(f'PDO: {PDO}')
print(f'Scale Factor: {FACTOR:.2f}')
print(f'\nScorecard:')
scorecard.head(20)

In [ ]:
# --- Calculate credit scores for test set ---
credit_scores = base_points + X_test_scaled @ (-coefficients * FACTOR / n_features)

score_df = pd.DataFrame({
    'credit_score': credit_scores,
    'actual_default': y_test
})

print(f'Score range: {credit_scores.min():.0f} - {credit_scores.max():.0f}')
print(f'Mean score (non-default): {score_df[score_df.actual_default == 0]["credit_score"].mean():.0f}')
print(f'Mean score (default):     {score_df[score_df.actual_default == 1]["credit_score"].mean():.0f}')

In [ ]:
# --- Score distribution: Good vs Bad ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overlapping histograms
for label, color, name in zip([0, 1], ['#2ecc71', '#e74c3c'], ['Non-Default', 'Default']):
    subset = score_df[score_df.actual_default == label]['credit_score']
    axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)

axes[0].set_title('Credit Score Distribution: Good vs Bad')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Density')
axes[0].legend()

# Default rate by score band
score_df['score_band'] = pd.qcut(score_df['credit_score'], 10, duplicates='drop')
band_stats = score_df.groupby('score_band', observed=True).agg(
    count=('actual_default', 'count'),
    default_rate=('actual_default', 'mean')
).reset_index()

axes[1].bar(
    range(len(band_stats)), band_stats['default_rate'],
    color='#e74c3c', edgecolor='black', alpha=0.8
)
ax2 = axes[1].twinx()
ax2.plot(
    range(len(band_stats)), band_stats['count'],
    'o-', color='#2980b9', lw=2, label='Count'
)
axes[1].set_title('Default Rate & Volume by Score Band')
axes[1].set_xlabel('Score Band (Low to High)')
axes[1].set_ylabel('Default Rate', color='#e74c3c')
ax2.set_ylabel('Count', color='#2980b9')
axes[1].set_xticks(range(len(band_stats)))
axes[1].set_xticklabels(
    [str(x)[:10] for x in band_stats['score_band']], rotation=45, fontsize=8
)

plt.tight_layout()
plt.show()

## Conclusiones

### Lo que encontramos

1. **Performance de los modelos:** Los tres modelos logran buena discriminacion, con los metodos de gradient boosting (LightGBM, XGBoost) superando a la regresion logistica. El KS statistic y el coeficiente de Gini confirman una buena capacidad de ordenamiento. En mi experiencia, esta diferencia de 2-3 puntos de AUC entre regresion logistica y gradient boosting es tipica en credit scoring.

2. **Drivers de riesgo principales:** El ratio deuda-ingreso, el numero de morosidades recientes y la utilizacion de credito son las variables mas predictivas en todos los modelos. Esto tiene sentido desde el punto de vista del negocio -- alguien que ya esta endeudado y ha fallado en pagos recientes tiene mayor probabilidad de defaultear.

3. **Viabilidad del scorecard:** El modelo de regresion logistica se traduce bien a un scorecard basado en puntos. Las bandas de puntaje muestran tasas de default monotonicamente crecientes, lo cual valida que el scorecard ordena correctamente el riesgo.

4. **Calibracion:** Los modelos de gradient boosting producen probabilidades mejor calibradas. La regresion logistica con pesos de clase balanceados podria necesitar recalibracion antes de ir a produccion.

### Recomendaciones para produccion

- **Para reguladores:** Usar el scorecard de regresion logistica por su transparencia
- **Motor de decision:** Desplegar LightGBM como modelo challenger para scoring en tiempo real
- **Monitoreo:** Hacer seguimiento del PSI (Population Stability Index) mensualmente para detectar data drift -- si la poblacion de solicitantes cambia, el modelo se degrada
- **Punto de corte:** Usar el lift chart para definir umbrales de aprobacion segun el apetito de riesgo del negocio